# How to add a new decoder to LightStim

LightStim's decoder backend is a small **registry** of [`sinter.Decoder`](https://github.com/quantumlib/Stim/tree/main/glue/sample) subclasses. Each decoder is registered under a *name* + *backend* (`cpu` / `gpu` / `fpga`), and at runtime

```python
DecoderConfig(name="…", backend="…", params={…})
```

looks it up. Every consumer — the simulation pipeline, the HTTP server, the notebooks, the benchmark scripts — goes through `DecoderConfig`, so once a decoder is registered it is usable *everywhere* by name.

This tutorial walks through **three integration patterns**, with one runnable example each. Pick the one that matches your decoder:

| Pattern | When to use | Example here |
|---|---|---|
| **A. Register a sinter decoder** | Upstream already implements `sinter.Decoder` | PyMatching (built-in) · **Relay-BP** |
| **B. Custom decoder from a DEM** | Not sinter-compatible; you want full control | **Tesseract** (shown as custom) |
| **C. `ExternalDecoder` facade** | Not sinter-compatible; you want the easy path | ldpc BP (correction) · Tesseract (observables) |

> The code cells are **defensive**: if an optional decoder library isn't installed (or a decoder errors), the cell prints why and continues, so the notebook runs end-to-end in any environment. PyMatching is a hard dependency; **Relay-BP**, **ldpc**, and **Tesseract** are optional. (A prebuilt Tesseract wheel may not match every CPU; if `import tesseract_decoder` fails, build it from source.)


## The contract

Every decoder is a `sinter.Decoder` that implements `compile_decoder_for_dem(dem) -> sinter.CompiledDecoder`, and the compiled decoder implements `decode_shots_bit_packed(...)` — **bit-packed detector data in, bit-packed observable predictions out** (little-endian / LSB-first).

LightStim adds three helpers around that:

- **`register_decoder(name, cls, aliases=[…], backend="cpu")`** — put a decoder class in the registry.
- **`list_decoders()`** — the registered (canonical) names.
- **`DecoderConfig` / `SimulationPipeline`** — the user-facing run API.

Let's set up a demo experiment and a tiny benchmark helper we'll reuse for every pattern.


In [2]:
# --- make 'lightstim' importable from a source checkout ---
# (no-op if you've run `pip install -e .` in this kernel's environment)
import pathlib
import sys

try:
    import lightstim  # noqa: F401
except ModuleNotFoundError:
    for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_p / "lightstim" / "__init__.py").exists():
            sys.path.insert(0, str(_p))
            break
    else:
        raise

import math
import numpy as np
import sinter
import stim

from lightstim.simulation.decoder_backend import (
    DecoderConfig,
    SimulationPipeline,
    list_decoders,
)
from lightstim.simulation.decoder_backend.registry import register_decoder

print("decoders registered at import:", list_decoders())


def demo_circuit(distance: int = 5, p: float = 1e-3) -> stim.Circuit:
    """A rotated surface-code Z-memory experiment with circuit-level noise.

    Used for Patterns A & B, whose decoders (PyMatching, BP+OSD, Relay-BP)
    build their graph / check matrix straight from the DEM and so handle the
    surface code's hyperedges. NOTE: at d=5, p=1e-3 the logical error rate is
    tiny (well below 1e-4), so a short run usually sees 0 errors - raise
    max_shots for a real estimate.
    """
    return stim.Circuit.generated(
        "surface_code:rotated_memory_z",
        distance=distance,
        rounds=distance,
        after_clifford_depolarization=p,
        before_round_data_depolarization=p,
        before_measure_flip_probability=p,
        after_reset_flip_probability=p,
    )


def benchmark(name, params=None, circuit=None, max_shots=200_000, **pipeline_kw):
    """Run one decoder through the pipeline and print its logical error rate.

    Defensive: skips gracefully if the decoder isn't registered or errors out.
    """
    if name not in list_decoders():
        print(f"  {name:14s} not registered (optional library missing) - skipped")
        return None
    try:
        stats = SimulationPipeline(
            decoder_config=DecoderConfig(name=name, params=params or {}),
            max_shots=max_shots,
            max_errors=100,
            batch_size=5_000,
            num_workers=4,
            print_progress=False,
            **pipeline_kw,
        ).run(circuit if circuit is not None else demo_circuit())
        print(
            f"  {name:14s} LER={stats.logical_error_rate:.3e}  "
            f"({stats.errors}/{stats.post_selected_shots} shots)"
        )
        return stats
    except Exception as exc:
        print(f"  {name:14s} unavailable: {type(exc).__name__}: {exc}")
        return None


decoders registered at import: ['bposd', 'mwpf', 'pymatching', 'relay-bp']


## Pattern A — register a sinter-native decoder

**When:** the upstream library already implements `sinter.Decoder`. There is nothing to write — you just put the class in the registry under a LightStim name.

### A1. PyMatching (built-in)

PyMatching ships as a `sinter.Decoder` and LightStim registers it out of the box. Use it by name:


In [3]:
benchmark("pymatching")


  pymatching     LER=2.500e-04  (50/200000 shots)


SimulationStats(shots=200000, post_selected_shots=200000, errors=50, seconds=0.5259534910146613, decoder='pymatching', json_metadata={})

### A2. Relay-BP — the highlighted example

[Relay-BP](https://github.com/trmue/relay) ships sinter-compatible decoders via `relay_bp.stim`. `SinterDecoder_RelayBP` **is** a `sinter.Decoder` (it implements both `compile_decoder_for_dem` and `decode_via_files`), so we register the class **directly** — the purest Pattern A, exactly like the built-in `mwpf` decoder.

`sinter_decoders(**kw)` returns `{"relay-bp": SinterDecoder_RelayBP(...), "mem-bp": ..., "msl-bp": ...}`; here we want the class itself so LightStim can construct it per-`DecoderConfig`.

> LightStim also ships this as a built-in module (`decoders/relay_bp.py`), so if `relay_bp` is installed it's *already* registered as `"relay-bp"`. The cell below shows the **user-side** path — registering it from your own code, no library edit needed.


In [4]:
# install relay-bp with: pip install relay_bp[stim]
try:
    from relay_bp.stim import SinterDecoder_RelayBP

    register_decoder(
        "relay-bp",
        SinterDecoder_RelayBP,
        aliases=["relay_bp", "relaybp"],
        backend="cpu",
    )
    print("registered 'relay-bp':", "relay-bp" in list_decoders())
except ImportError:
    print("relay_bp not installed - skipping registration")

# Relay-BP knobs flow straight through DecoderConfig(params=...) to
# SinterDecoder_RelayBP. gamma_dist_interval is sensitive - tune per code/DEM.
benchmark(
    "relay-bp",
    params={
        "gamma0": 0.1,
        "pre_iter": 80,
        "num_sets": 60,
        "set_max_iter": 60,
        "gamma_dist_interval": (-0.24, 0.66),
        "stop_nconv": 1,
    },
)


registered 'relay-bp': True
  relay-bp       LER=1.450e-04  (29/200000 shots)


SimulationStats(shots=200000, post_selected_shots=200000, errors=29, seconds=9.036197791981976, decoder='relay-bp', json_metadata={})

**Even simpler:** if the upstream library is *fully* sinter-native (class instantiable with no required args), registration is a one-liner with no example circuit needed — e.g. the built-in MWPF decoder is literally:

```python
from mwpf import SinterMWPFDecoder
register_decoder("mwpf", SinterMWPFDecoder)
```

**Need different param names or defaults?** Give your wrapper a custom `__init__(**params)` that renames or defaults kwargs before constructing the inner decoder — registration is otherwise identical. That's all the built-in `bposd` wrapper does, to share one parameter vocabulary across its CPU and GPU BP+OSD backends. For a brand-new decoder you usually *don't* need this — just register it and let users pass its native params.


## Pattern B — custom decoder from a DEM (full control)

**When:** the upstream library is **not** sinter-compatible and you want to own the `sinter.CompiledDecoder` lifecycle yourself — build your decoder from the DEM (or its matrices) and write the unpack → decode → pack loop.

We illustrate it with [**Tesseract**](https://github.com/quantumlib/tesseract-decoder), Google's beam-search most-likely-error decoder. It builds straight from a `stim.DetectorErrorModel` and returns **observable flips** directly (no correction-matrix multiply needed):

```python
config  = tesseract.TesseractConfig(dem=dem, det_beam=50)
decoder = config.compile_decoder()
flipped_observables = decoder.decode(syndrome)   # bool[n_dets] -> bool[n_obs]
```

> **Note — Tesseract is actually sinter-native.** It ships `TesseractSinterDecoder` and `make_tesseract_sinter_decoders_dict`, so in real use you'd register it with **Pattern A** in one line (just like Relay-BP). We deliberately use its lower-level `compile_decoder()` / `decode()` API here *only to illustrate Pattern B* — wrapping a non-sinter decoder in your own `CompiledDecoder`.

The three parts: (1) build the decoder from the DEM, (2) a custom `CompiledDecoder` that unpacks the syndromes, loops `decode`, and packs the observable flips, (3) a top-level `sinter.Decoder`. Since Tesseract is hyperedge-aware, this runs on the surface-code DEM directly (no graphlike restriction).


In [8]:
try:
    from tesseract_decoder import tesseract

    class _TesseractCompiled(sinter.CompiledDecoder):
        def __init__(self, decoder, n_dets, n_obs):
            self._d = decoder
            self._n_dets = n_dets
            self._n_obs = n_obs

        def decode_shots_bit_packed(self, *, bit_packed_detection_event_data):
            # unpack -> (n_shots, n_dets) bool
            syndromes = np.unpackbits(
                bit_packed_detection_event_data, axis=1, bitorder="little"
            )[:, : self._n_dets].astype(bool)
            # Tesseract decodes one syndrome at a time, returning observable flips.
            preds = np.array(
                [self._d.decode(s) for s in syndromes], dtype=np.uint8
            ).reshape(len(syndromes), self._n_obs)
            # pack back, trimmed to the exact observable-byte width
            return np.packbits(preds, axis=1, bitorder="little")[:, : math.ceil(self._n_obs / 8)]

    class TesseractDecoder(sinter.Decoder):
        def __init__(self, **params):
            self._params = params                       # e.g. det_beam, beam_climbing

        def compile_decoder_for_dem(self, *, dem):
            config = tesseract.TesseractConfig(dem=dem, **self._params)
            return _TesseractCompiled(
                config.compile_decoder(), dem.num_detectors, dem.num_observables
            )

    register_decoder("tesseract", TesseractDecoder)
    print("registered 'tesseract':", "tesseract" in list_decoders())
except ImportError:
    print("tesseract_decoder not installed - skipping")

# Tesseract is a beam-search MLE decoder (slower per shot), so keep the shot count modest.
benchmark("tesseract", params={"det_beam": 50})


registered 'tesseract': True
  tesseract      LER=6.000e-05  (12/200000 shots)


SimulationStats(shots=200000, post_selected_shots=200000, errors=12, seconds=45.589343569998164, decoder='tesseract', json_metadata={})

## Pattern C — the `ExternalDecoder` facade (recommended for non-sinter decoders)

**When:** same as Pattern B, but you'd rather *not* touch bit-packing, the sinter classes, or the worker plumbing.

Subclass `ExternalDecoder`, declare `output_type`, build your decoder in `setup` (you receive the `dem` and its matrices `H`, `obs_matrix`, `priors`), and implement **one** of `decode_single` / `decode_batch` (LightStim bridges the other) on plain **unpacked** arrays.

`output_type` is the key knob — it declares *what your decode returns*:

- **`"correction"`** — a correction over the *error mechanisms* (length `n_err`); LightStim multiplies it by the observable matrix to get the logical flips. Natural for BP / matching-style decoders.
- **`"observables"`** — the logical-observable flips directly (length `n_obs`); no matrix multiply. Natural for MLE / neural / lookup decoders.

We show one of each.

### Example 1 — `output_type="correction"`: plain BP from `ldpc`

[`ldpc`](https://github.com/quantumgizmos/ldpc)'s `BpDecoder` is a non-sinter belief-propagation decoder: build it from the check matrix `H` + priors, and `decode(syndrome)` returns a correction over error mechanisms. BP can **fail to converge** — we pass its `.converge` flag straight to LightStim (see *Failure flags* below). BP handles the surface code's hyperedges, so it runs on the demo DEM directly.


In [5]:
from lightstim.simulation.decoder_backend.external import ExternalDecoder

try:
    from ldpc import BpDecoder

    class BpExternal(ExternalDecoder):
        # Return a correction over error mechanisms; LightStim multiplies by the
        # observable matrix for us.
        output_type = "correction"

        def setup(self, *, H, priors, **_):
            self._bp = BpDecoder(
                H, error_channel=list(priors), max_iter=100, bp_method="min_sum"
            )

        def decode_single(self, syndrome):
            correction = self._bp.decode(syndrome)
            # BP may not converge - pass the convergence flag (False = failed).
            return correction.astype(np.uint8), bool(self._bp.converge)

    register_decoder("ldpc-bp", BpExternal)
    print("registered 'ldpc-bp':", "ldpc-bp" in list_decoders())
except ImportError:
    print("ldpc not installed - skipping")

# Plain BP (no OSD) is a weak surface-code decoder and often fails to converge;
# with on_decode_failure="error" (default) those shots count as logical errors.
benchmark("ldpc-bp")


registered 'ldpc-bp': True
  ldpc-bp        LER=3.155e-02  (631/20000 shots)


SimulationStats(shots=20000, post_selected_shots=20000, errors=631, seconds=1.5319177219935227, decoder='ldpc-bp', json_metadata={})

### Example 2 — `output_type="observables"`: Tesseract

[Tesseract](https://github.com/quantumlib/tesseract-decoder) returns **observable flips** directly, so `output_type="observables"` and there's no matrix multiply. It's the *same* decoder as Pattern B — but via the facade it's a few lines, no hand-written `CompiledDecoder`.


In [9]:
try:
    from tesseract_decoder import tesseract

    class TesseractExternal(ExternalDecoder):
        # Tesseract returns observable flips directly (not a correction over errors).
        output_type = "observables"

        def setup(self, *, dem, **_):
            # self.params holds DecoderConfig(params=...), e.g. det_beam.
            self._d = tesseract.TesseractConfig(dem=dem, **self.params).compile_decoder()

        def decode_single(self, syndrome):
            return self._d.decode(syndrome.astype(bool)).astype(np.uint8), None

    register_decoder("tesseract-ext", TesseractExternal)
    print("registered 'tesseract-ext':", "tesseract-ext" in list_decoders())
except ImportError:
    print("tesseract_decoder not installed - skipping")

# Same predictions as the Pattern B 'tesseract' decoder - the facade owns the packing.
benchmark("tesseract-ext", params={"det_beam": 50})


registered 'tesseract-ext': True
  tesseract-ext  LER=6.000e-05  (12/200000 shots)


SimulationStats(shots=200000, post_selected_shots=200000, errors=12, seconds=45.58869765899726, decoder='tesseract-ext', json_metadata={})

### Failure flags (`on_decode_failure`)

The `ldpc-bp` decoder above returns a real per-shot `converge` flag (`False` = BP didn't converge). What LightStim does with a `False` is a `DecoderConfig` policy:

- `"error"` *(default)* — count the shot as a logical error,
- `"discard"` — herald it: drop the shot from the denominator,
- `"ignore"` — trust the returned prediction anyway.

(Return `None`/`True` to mean "always succeeded", as the Tesseract example does.) Here's the *same* `ldpc-bp` decoder under all three policies — note how `discard` reveals that BP, *when it converges*, is essentially error-free at this noise level:


In [6]:
if "ldpc-bp" not in list_decoders():
    print("ldpc not installed - skipping failure-flag demo")
else:
    for policy in ("error", "discard", "ignore"):
        stats = SimulationPipeline(
            decoder_config=DecoderConfig("ldpc-bp", on_decode_failure=policy),
            max_shots=20_000, max_errors=10**9, batch_size=5_000,
            num_workers=1, print_progress=False,
        ).run(demo_circuit())
        print(
            f"  on_decode_failure={policy:8s} -> kept={stats.post_selected_shots:6d} "
            f"errors={stats.errors:5d} LER={stats.logical_error_rate:.3e}"
        )


  on_decode_failure=error    -> kept= 20000 errors=  667 LER=3.335e-02
  on_decode_failure=discard  -> kept= 19333 errors=    0 LER=0.000e+00
  on_decode_failure=ignore   -> kept= 20000 errors=   57 LER=2.850e-03


## Making it permanent

The cells above register decoders *at runtime* from the notebook — perfect for experiments. To ship a decoder as a **built-in** (usable by name everywhere, including the server and benchmarks), add it to the library:

1. **One new file** in `lightstim/simulation/decoder_backend/decoders/`, ending with a `register_decoder(...)` call (wrap the upstream import in `try/except ImportError`).
2. **One soft-import hook** in `decoders/__init__.py`, behind an `importlib.util.find_spec("your_lib")` guard, so users without the library don't hit an `ImportError` at startup.
3. **One smoke test** in `tests/test_simulation_backend_quality.py` (gate optional deps with `pytest.importorskip`).

The built-in Relay-BP integration (`decoders/relay_bp.py`) is exactly this — ~15 lines.

### Where things live

| File | What it is |
|---|---|
| `decoder_backend/registry.py` | `register_decoder()`, `get_decoder()`, `list_decoders()` |
| `decoder_backend/decoders/__init__.py` | soft-import dispatcher |
| `decoder_backend/decoders/{pymatching,mwpf,relay_bp}.py` | Pattern A references |
| `decoder_backend/decoders/bposd.py` | sinter wrapper that renames params (see Pattern A note) |
| `decoder_backend/decoders/cudaqx.py` | Pattern B reference (DEM matrix + GPU) |
| `decoder_backend/external.py` | Pattern C — `ExternalDecoder` facade |
| `decoder_backend/dem_matrices.py` | shared `dem_to_matrices()` |
| `decoder_backend/config.py` | `DecoderConfig` (incl. `on_decode_failure`) |

### Gotchas

- **Bit order is little-endian** (`bitorder="little"`) for both syndromes in and predictions out. Pattern C handles this for you.
- **Failure flags** ride an in-process side channel and require LightStim's `SimulationPipeline`, not a raw `sinter.collect`.

See `skills/extend-new-decoder/SKILL.md` for the full reference.
